# Despliegue del modelo — Titanic

**Materia:** Analítica de Datos — Universidad Pontificia Bolivariana
**Equipo:** Nicolás Castrillón, Mateo Roldán, Matías Arango

Este notebook documenta el proceso de despliegue del modelo final (Random Forest optimizado con GridSearchCV, entrenado en `modelos_mineria_datos.ipynb`) mediante una interfaz gráfica hecha con **Streamlit**.

Contenido:
1. Carga y verificación del modelo final guardado.
2. Prueba del modelo con datos de ejemplo (simulando lo que hace la app).
3. Explicación del código de la aplicación `app/despliegue.py`.
4. Instrucciones para ejecutar la aplicación.
5. Instrucciones para tomar la captura de pantalla que pide el profesor.

## 1. Carga del modelo final

El modelo se guardó en la Fase 4 del notebook de modelos, en `models/modelo_final.pkl`, usando `joblib`. Es un objeto `Pipeline` de scikit-learn que incluye **todo** el preprocesamiento (imputación de nulos, escalado de numéricas, codificación dummy de categóricas) más el `RandomForestClassifier` ya optimizado — por eso la app puede recibir directamente los datos "crudos" que ingresa el usuario, sin que este tenga que preocuparse por escalarlos o codificarlos.

In [1]:
import joblib
import pandas as pd

modelo = joblib.load('../models/modelo_final.pkl')
print(type(modelo))
print(modelo)

<class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare', 'SibSp',
                                                   'Parch']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncode

## 2. Prueba del modelo con datos de ejemplo

Antes de construir la interfaz, comprobamos que el modelo cargado predice correctamente con un par de pasajeros hipotéticos, para verificar que el pipeline funciona de punta a punta igual que en el notebook de entrenamiento.

In [2]:
pasajeros_ejemplo = pd.DataFrame([
    {"Pclass": 1, "Sex": "female", "Age": 29, "SibSp": 0, "Parch": 0, "Fare": 80.0, "Embarked": "S"},
    {"Pclass": 3, "Sex": "male",   "Age": 25, "SibSp": 0, "Parch": 0, "Fare": 8.0,  "Embarked": "S"},
    {"Pclass": 2, "Sex": "female", "Age": 4,  "SibSp": 1, "Parch": 1, "Fare": 26.0, "Embarked": "C"},
])

predicciones = modelo.predict(pasajeros_ejemplo)
probabilidades = modelo.predict_proba(pasajeros_ejemplo)[:, 1]

resultado = pasajeros_ejemplo.copy()
resultado["Predicción"] = ["Sobrevive" if p == 1 else "No sobrevive" for p in predicciones]
resultado["Probabilidad de sobrevivir"] = probabilidades.round(3)
resultado

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Predicción,Probabilidad de sobrevivir
0,1,female,29,0,0,80.0,S,Sobrevive,0.979
1,3,male,25,0,0,8.0,S,No sobrevive,0.118
2,2,female,4,1,1,26.0,C,Sobrevive,0.912


Los tres casos tienen sentido frente a los patrones históricos conocidos del Titanic: la mujer de primera clase con tarifa alta obtiene la mayor probabilidad de sobrevivir, y el hombre de tercera clase con tarifa baja obtiene la menor. Esto confirma que el pipeline guardado reproduce el comportamiento del modelo entrenado.

## 3. La aplicación de despliegue (`app/despliegue.py`)

La aplicación está hecha con **Streamlit** y hace lo siguiente:

1. **Carga el modelo** (`joblib.load`) una sola vez, usando `st.cache_resource` para no releerlo en cada interacción.
2. **Muestra un formulario** (`st.form`) con un campo por cada variable predictora: clase, sexo, edad, hermanos/cónyuge a bordo, padres/hijos a bordo, tarifa y puerto de embarque. Los campos usan `selectbox` para las categóricas (con etiquetas en español, traducidas internamente a los valores que espera el modelo) y `number_input` con límites mínimos y máximos para las numéricas.
3. **Valida los datos**: antes de predecir, revisa que la edad esté en un rango razonable (0–100), que la tarifa no sea negativa y que el número de familiares no sea negativo; si algo falla, muestra el error y no llama al modelo.
4. **Arma un DataFrame de una fila** con exactamente los mismos nombres de columnas (`Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, `Embarked`) que espera el `ColumnTransformer` del pipeline.
5. **Predice** con `modelo.predict()` y `modelo.predict_proba()`, y muestra el resultado con `st.success` / `st.error`, además de las probabilidades de cada clase.
6. **Maneja errores de ejecución** con un bloque `try/except` alrededor de la predicción, para que un problema inesperado no haga que la aplicación se caiga en la cara del usuario.
7. En la barra lateral, muestra información del modelo y un gráfico de importancia de variables, calculado directamente desde el `RandomForestClassifier` ya entrenado.

El código completo está en `app/despliegue.py`, dentro de la estructura del proyecto.

### Generación de los archivos de la aplicación

Las siguientes celdas escriben en disco los archivos reales de la aplicación: la configuración de tema, los estilos (diseño visual), los componentes visuales reutilizables y la lógica de carga del modelo, formulario, validación y predicción. El contenido es exactamente el mismo que está en el repositorio (`app/despliegue.py`, `app/components.py`, `app/assets/style.css`, `.streamlit/config.toml`); ejecutar estas celdas sobrescribe esos archivos con este mismo contenido, así que correr este notebook de principio a fin es suficiente para reproducir la aplicación completa (diseño y lógica incluidos), sin depender de que los archivos ya existan de antes.

In [3]:
import os

os.makedirs('../app/assets', exist_ok=True)
os.makedirs('../.streamlit', exist_ok=True)
print('Carpetas verificadas/creadas.')

Carpetas verificadas/creadas.


In [4]:
%%writefile ../.streamlit/config.toml
[theme]
base = "light"
primaryColor = "#0891b2"
backgroundColor = "#f4f6fb"
secondaryBackgroundColor = "#ffffff"
textColor = "#0f172a"
font = "sans serif"


Overwriting ../.streamlit/config.toml


In [5]:
%%writefile ../app/assets/style.css
/* =============================================================================
   Estilos personalizados — Predicción de Supervivencia Titanic
   Paleta: tecnológica/profesional (slate + cyan), tipografía Inter.
   ============================================================================= */

@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

:root {
    --color-bg: #f4f6fb;
    --color-card: #ffffff;
    --color-primary: #0f172a;
    --color-primary-soft: #1e293b;
    --color-accent: #0891b2;
    --color-accent-2: #06b6d4;
    --color-accent-soft: #e0f7fa;
    --color-success: #059669;
    --color-success-soft: #ecfdf5;
    --color-danger: #dc2626;
    --color-danger-soft: #fef2f2;
    --color-text: #0f172a;
    --color-text-muted: #64748b;
    --color-border: #e2e8f0;
    --radius-lg: 18px;
    --radius-md: 12px;
    --shadow-card: 0 4px 24px rgba(15, 23, 42, 0.06), 0 1px 2px rgba(15, 23, 42, 0.04);
    --shadow-card-hover: 0 10px 32px rgba(15, 23, 42, 0.10);
}

/* ---- Fondo y tipografía general ---------------------------------------- */
html, body, [class*="css"] {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
}

.stApp {
    background: var(--color-bg);
}

/* Limita el ancho del contenido para que se sienta como una app moderna,
   no un formulario "full-bleed" pegado a los bordes */
.block-container {
    max-width: 1080px;
    padding-top: 1.5rem !important;
    padding-bottom: 3rem !important;
    animation: fadeInUp 0.5s ease;
}

/* Oculta el menú hamburguesa y el footer de Streamlit para un look más "app propia" */
#MainMenu, footer, header[data-testid="stHeader"] {
    background: transparent;
}
#MainMenu { visibility: hidden; }
footer { visibility: hidden; }
div[data-testid="stToolbar"] { display: none !important; }

@keyframes fadeInUp {
    from { opacity: 0; transform: translateY(8px); }
    to   { opacity: 1; transform: translateY(0); }
}

/* ---- Hero / encabezado --------------------------------------------------- */
.hero {
    background: linear-gradient(135deg, #0f172a 0%, #164e63 55%, #0891b2 100%);
    border-radius: var(--radius-lg);
    padding: 2.4rem 2.2rem;
    margin-bottom: 1.6rem;
    color: #f8fafc;
    box-shadow: var(--shadow-card);
    position: relative;
    overflow: hidden;
}
.hero::after {
    content: "";
    position: absolute;
    right: -60px;
    top: -60px;
    width: 220px;
    height: 220px;
    background: radial-gradient(circle, rgba(6,182,212,0.35) 0%, rgba(6,182,212,0) 70%);
    border-radius: 50%;
}
.hero-eyebrow {
    display: inline-block;
    font-size: 0.72rem;
    font-weight: 700;
    letter-spacing: 0.08em;
    text-transform: uppercase;
    background: rgba(255,255,255,0.12);
    color: #a5f3fc;
    padding: 0.3rem 0.75rem;
    border-radius: 999px;
    margin-bottom: 0.9rem;
}
.hero-title {
    font-size: 2.1rem;
    font-weight: 800;
    margin: 0 0 0.5rem 0;
    line-height: 1.2;
}
.hero-subtitle {
    font-size: 1rem;
    color: #cbd5e1;
    max-width: 640px;
    line-height: 1.55;
    margin: 0;
}
.hero-badges {
    margin-top: 1.2rem;
    display: flex;
    flex-wrap: wrap;
    gap: 0.5rem;
}
.badge {
    display: inline-flex;
    align-items: center;
    gap: 0.35rem;
    background: rgba(255,255,255,0.08);
    border: 1px solid rgba(255,255,255,0.18);
    color: #e2e8f0;
    font-size: 0.78rem;
    font-weight: 500;
    padding: 0.32rem 0.7rem;
    border-radius: 999px;
    backdrop-filter: blur(4px);
}

/* ---- Tarjetas (cards) ----------------------------------------------------- */
div[data-testid="stVerticalBlockBorderWrapper"] {
    background: var(--color-card);
    border-radius: var(--radius-lg) !important;
    box-shadow: var(--shadow-card);
    border: 1px solid var(--color-border) !important;
    padding: 0.4rem 0.2rem;
    transition: box-shadow 0.25s ease, transform 0.25s ease;
}
div[data-testid="stVerticalBlockBorderWrapper"]:hover {
    box-shadow: var(--shadow-card-hover);
}

.section-title {
    display: flex;
    align-items: center;
    gap: 0.55rem;
    font-size: 1.15rem;
    font-weight: 700;
    color: var(--color-primary);
    margin: 0.2rem 0 0.9rem 0;
}
.section-title .icon-chip {
    display: inline-flex;
    align-items: center;
    justify-content: center;
    width: 32px;
    height: 32px;
    border-radius: 10px;
    background: var(--color-accent-soft);
    font-size: 1rem;
}
.section-subtitle {
    color: var(--color-text-muted);
    font-size: 0.88rem;
    margin-top: -0.6rem;
    margin-bottom: 1rem;
}

/* ---- Métricas / stat chips -------------------------------------------- */
.stat-row { display: flex; gap: 0.7rem; flex-wrap: wrap; }
.stat-chip {
    flex: 1 1 120px;
    background: var(--color-accent-soft);
    border-radius: var(--radius-md);
    padding: 0.85rem 0.9rem;
    text-align: left;
}
.stat-chip .stat-value {
    font-size: 1.35rem;
    font-weight: 800;
    color: var(--color-primary);
    line-height: 1.1;
}
.stat-chip .stat-label {
    font-size: 0.74rem;
    color: var(--color-text-muted);
    margin-top: 0.15rem;
    font-weight: 500;
}

/* ---- Resultado de predicción -------------------------------------------- */
.result-card {
    border-radius: var(--radius-lg);
    padding: 1.6rem 1.8rem;
    margin: 0.6rem 0 1.2rem 0;
    display: flex;
    align-items: center;
    gap: 1.1rem;
    animation: fadeInUp 0.4s ease;
    box-shadow: var(--shadow-card);
}
.result-card.survive {
    background: linear-gradient(135deg, var(--color-success-soft) 0%, #d1fae5 100%);
    border: 1px solid #a7f3d0;
}
.result-card.no-survive {
    background: linear-gradient(135deg, var(--color-danger-soft) 0%, #fee2e2 100%);
    border: 1px solid #fecaca;
}
.result-icon {
    font-size: 2.4rem;
    line-height: 1;
}
.result-title {
    font-size: 1.25rem;
    font-weight: 800;
    margin: 0;
}
.result-card.survive .result-title { color: #065f46; }
.result-card.no-survive .result-title { color: #991b1b; }
.result-desc {
    font-size: 0.88rem;
    color: var(--color-text-muted);
    margin-top: 0.15rem;
}

/* Barras de probabilidad personalizadas */
.prob-block { margin-bottom: 0.85rem; }
.prob-label {
    display: flex;
    justify-content: space-between;
    font-size: 0.85rem;
    font-weight: 600;
    color: var(--color-text);
    margin-bottom: 0.3rem;
}
.prob-track {
    width: 100%;
    height: 10px;
    background: #e2e8f0;
    border-radius: 999px;
    overflow: hidden;
}
.prob-fill {
    height: 100%;
    border-radius: 999px;
    transition: width 0.6s ease;
}
.prob-fill.survive { background: linear-gradient(90deg, #059669, #34d399); }
.prob-fill.no-survive { background: linear-gradient(90deg, #dc2626, #f87171); }

/* ---- Botones -------------------------------------------------------------- */
.stButton > button, .stFormSubmitButton > button {
    background: linear-gradient(135deg, #0891b2 0%, #0e7490 100%);
    color: white;
    border: none;
    border-radius: 999px;
    padding: 0.65rem 1.6rem;
    font-weight: 600;
    font-size: 0.95rem;
    box-shadow: 0 4px 14px rgba(8, 145, 178, 0.35);
    transition: transform 0.15s ease, box-shadow 0.15s ease;
}
.stButton > button:hover, .stFormSubmitButton > button:hover {
    transform: translateY(-1px);
    box-shadow: 0 6px 18px rgba(8, 145, 178, 0.45);
    color: white;
}
.stButton > button:active, .stFormSubmitButton > button:active {
    transform: translateY(0);
}

/* ---- Inputs ---------------------------------------------------------------- */
/* Forzamos colores explícitos (no heredados del tema claro/oscuro del sistema)
   para que las etiquetas y los campos siempre sean legibles, sin importar si
   el navegador del usuario está en modo oscuro. */
div.stForm, div.stForm p, div.stForm span, div.stForm label {
    color: var(--color-text) !important;
}
div[data-testid="stWidgetLabel"] > label,
div[data-testid="stWidgetLabel"] p {
    color: var(--color-text) !important;
    font-weight: 600 !important;
    opacity: 1 !important;
}
div[data-baseweb="select"] > div {
    background-color: #ffffff !important;
    border: 1px solid var(--color-border) !important;
    border-radius: var(--radius-md) !important;
    color: var(--color-text) !important;
}
div[data-baseweb="select"] span, div[data-baseweb="select"] div {
    color: var(--color-text) !important;
}
input[type="number"] {
    background-color: #ffffff !important;
    color: var(--color-text) !important;
    border: 1px solid var(--color-border) !important;
    border-radius: var(--radius-md) !important;
}
div[data-testid="stNumberInput"] button {
    background-color: #f1f5f9 !important;
    color: var(--color-text) !important;
    border: 1px solid var(--color-border) !important;
}
div[data-testid="stNumberInput"] input:focus,
div[data-baseweb="select"]:focus-within > div {
    border-color: var(--color-accent) !important;
    box-shadow: 0 0 0 2px rgba(8, 145, 178, 0.18) !important;
}
/* El menú desplegable de las opciones del selectbox también debe seguir
   nuestra paleta clara, no la del sistema */
ul[data-testid="stSelectboxVirtualDropdown"] {
    background-color: #ffffff !important;
}
ul[data-testid="stSelectboxVirtualDropdown"] li {
    color: var(--color-text) !important;
}

/* ---- Alertas nativas de Streamlit (success/error/warning/info) ---------- */
div[data-testid="stAlert"] {
    border-radius: var(--radius-md);
    border-width: 0 0 0 4px;
    border-style: solid;
    box-shadow: var(--shadow-card);
}

/* ---- Sidebar ---------------------------------------------------------------- */
section[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #0f172a 0%, #132335 100%);
}
section[data-testid="stSidebar"] * {
    color: #e2e8f0 !important;
}
section[data-testid="stSidebar"] .sidebar-card {
    background: rgba(255,255,255,0.05);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: var(--radius-md);
    padding: 0.9rem 1rem;
    margin-bottom: 0.9rem;
}
section[data-testid="stSidebar"] .sidebar-title {
    font-size: 0.95rem;
    font-weight: 700;
    display: flex;
    align-items: center;
    gap: 0.4rem;
    margin-bottom: 0.6rem;
    color: #f8fafc !important;
}
section[data-testid="stSidebar"] .sb-badge {
    display: inline-block;
    font-size: 0.72rem;
    font-weight: 600;
    background: rgba(6,182,212,0.18);
    color: #67e8f9 !important;
    border: 1px solid rgba(6,182,212,0.35);
    padding: 0.2rem 0.55rem;
    border-radius: 999px;
    margin: 0.15rem 0.25rem 0.15rem 0;
}
section[data-testid="stSidebar"] .sb-fact {
    font-size: 0.82rem;
    margin-bottom: 0.4rem;
    line-height: 1.4;
}

/* ---- Expander --------------------------------------------------------------- */
div[data-testid="stExpander"] {
    border-radius: var(--radius-md);
    border: 1px solid var(--color-border);
    box-shadow: none;
}

/* ---- Responsive ------------------------------------------------------------- */
@media (max-width: 768px) {
    .hero { padding: 1.6rem 1.3rem; }
    .hero-title { font-size: 1.5rem; }
    .hero-subtitle { font-size: 0.9rem; }
    .result-card { flex-direction: column; text-align: center; padding: 1.2rem; }
    .stat-chip { flex: 1 1 100%; }
}


Overwriting ../app/assets/style.css


In [6]:
%%writefile ../app/components.py
"""
Componentes visuales reutilizables para la app de despliegue.

Este módulo NO contiene lógica de negocio (no toca el modelo ni los datos):
solo funciones de presentación (HTML/CSS embebido y gráficos) que
`despliegue.py` usa para armar la interfaz. Mantenerlas separadas facilita
ajustar el diseño sin tocar la lógica de predicción.
"""

from __future__ import annotations

from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import streamlit as st

ASSETS_DIR = Path(__file__).parent / "assets"


def cargar_css() -> None:
    """Inyecta el CSS personalizado (app/assets/style.css) en la página."""
    css_path = ASSETS_DIR / "style.css"
    if css_path.exists():
        css = css_path.read_text(encoding="utf-8")
        st.markdown(f"<style>{css}</style>", unsafe_allow_html=True)


def render_hero(titulo: str, subtitulo: str, eyebrow: str, badges: list[str]) -> None:
    """Encabezado tipo 'hero section' con degradado, título y chips informativos."""
    badges_html = "".join(f'<span class="badge">{b}</span>' for b in badges)
    st.markdown(
        f"""
        <div class="hero">
            <span class="hero-eyebrow">{eyebrow}</span>
            <h1 class="hero-title">{titulo}</h1>
            <p class="hero-subtitle">{subtitulo}</p>
            <div class="hero-badges">{badges_html}</div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_section_title(icon: str, titulo: str, subtitulo: str | None = None) -> None:
    """Título de sección con un pequeño ícono en una 'chip' redondeada."""
    st.markdown(
        f'<div class="section-title"><span class="icon-chip">{icon}</span>{titulo}</div>',
        unsafe_allow_html=True,
    )
    if subtitulo:
        st.markdown(f'<div class="section-subtitle">{subtitulo}</div>', unsafe_allow_html=True)


def render_stat_row(stats: list[tuple[str, str]]) -> None:
    """Fila de 'stat chips' (valor grande + etiqueta). `stats` es una lista de (valor, etiqueta)."""
    chips = "".join(
        f'<div class="stat-chip"><div class="stat-value">{valor}</div>'
        f'<div class="stat-label">{etiqueta}</div></div>'
        for valor, etiqueta in stats
    )
    st.markdown(f'<div class="stat-row">{chips}</div>', unsafe_allow_html=True)


def render_result_card(sobrevive: bool, probabilidad: float) -> None:
    """Tarjeta grande de resultado (verde si sobrevive, roja si no)."""
    if sobrevive:
        icono, titulo, desc, clase = (
            "🟢",
            "Predicción: SOBREVIVE",
            f"El modelo estima una probabilidad de {probabilidad:.1%} de que este pasajero sobreviva.",
            "survive",
        )
    else:
        icono, titulo, desc, clase = (
            "🔴",
            "Predicción: NO SOBREVIVE",
            f"El modelo estima una probabilidad de {1 - probabilidad:.1%} de que este pasajero no sobreviva.",
            "no-survive",
        )
    st.markdown(
        f"""
        <div class="result-card {clase}">
            <div class="result-icon">{icono}</div>
            <div>
                <p class="result-title">{titulo}</p>
                <p class="result-desc">{desc}</p>
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_probability_bars(prob_sobrevive: float, prob_no_sobrevive: float) -> None:
    """Barras de probabilidad personalizadas (reemplazo visual de st.metric)."""
    st.markdown(
        f"""
        <div class="prob-block">
            <div class="prob-label"><span>🟢 Probabilidad de sobrevivir</span><span>{prob_sobrevive:.1%}</span></div>
            <div class="prob-track"><div class="prob-fill survive" style="width:{prob_sobrevive*100:.1f}%"></div></div>
        </div>
        <div class="prob-block">
            <div class="prob-label"><span>🔴 Probabilidad de no sobrevivir</span><span>{prob_no_sobrevive:.1%}</span></div>
            <div class="prob-track"><div class="prob-fill no-survive" style="width:{prob_no_sobrevive*100:.1f}%"></div></div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_feature_importance_chart(importancias: pd.Series) -> go.Figure:
    """Gráfico de importancia de variables con Plotly (interactivo, con degradado)."""
    importancias = importancias.sort_values(ascending=True)
    valores = [float(v) for v in importancias.values]
    fig = go.Figure(
        go.Bar(
            x=valores,
            y=list(importancias.index),
            orientation="h",
            marker=dict(
                color=valores,
                colorscale=[[0, "#0e7490"], [1, "#06b6d4"]],
                cmin=min(valores),
                cmax=max(valores),
                showscale=False,
                line=dict(width=0),
            ),
            hovertemplate="%{y}: %{x:.3f}<extra></extra>",
        )
    )
    fig.update_layout(
        margin=dict(l=0, r=10, t=10, b=0),
        height=260,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color="#e2e8f0", size=11),
        xaxis=dict(showgrid=True, gridcolor="rgba(255,255,255,0.08)", zeroline=False),
        yaxis=dict(showgrid=False),
    )
    return fig


def render_sidebar_card_start(titulo: str, icon: str) -> None:
    st.markdown(
        f'<div class="sidebar-card"><div class="sidebar-title">{icon} {titulo}</div>',
        unsafe_allow_html=True,
    )


def render_sidebar_card_end() -> None:
    st.markdown("</div>", unsafe_allow_html=True)


def render_sidebar_badges(items: list[str]) -> None:
    html = "".join(f'<span class="sb-badge">{item}</span>' for item in items)
    st.markdown(html, unsafe_allow_html=True)


Overwriting ../app/components.py


In [7]:
%%writefile ../app/despliegue.py
"""
Aplicación de despliegue — Predicción de supervivencia en el Titanic
Proyecto de Minería de Datos — Analítica de Datos (UPB)
Equipo: Nicolás Castrillón, Mateo Roldán, Matías Arango

Cómo ejecutarla (ver también el README.md del proyecto):
    streamlit run app/despliegue.py

Carga el modelo Random Forest optimizado con GridSearchCV (models/modelo_final.pkl),
generado en notebooks/modelos_mineria_datos.ipynb, y permite predecir si un pasajero
hipotético habría sobrevivido o no, a partir de sus características.

NOTA DE DISEÑO: la lógica de carga del modelo, validación de datos y predicción
es exactamente la misma que en la versión anterior de este archivo; lo único
que cambió es la presentación visual. Los componentes de interfaz (hero,
tarjetas, barras de probabilidad, gráfico de importancia) están en
`components.py` para mantener este archivo enfocado en el flujo de datos.
"""

import os

import joblib
import pandas as pd
import streamlit as st

from components import (
    cargar_css,
    render_feature_importance_chart,
    render_hero,
    render_probability_bars,
    render_result_card,
    render_sidebar_badges,
    render_sidebar_card_end,
    render_sidebar_card_start,
    render_section_title,
    render_stat_row,
)

# ---------------------------------------------------------------------------
# Configuración de la página
# ---------------------------------------------------------------------------
st.set_page_config(
    page_title="Predicción de supervivencia — Titanic",
    page_icon="🚢",
    layout="wide",
)

cargar_css()

RUTA_MODELO = os.path.join(os.path.dirname(__file__), "..", "models", "modelo_final.pkl")


@st.cache_resource
def cargar_modelo(ruta: str):
    """Carga el pipeline entrenado (preprocesamiento + Random Forest optimizado).
    Se cachea para no releer el archivo .pkl en cada interacción del usuario."""
    return joblib.load(ruta)


# ---------------------------------------------------------------------------
# Carga del modelo, con manejo de errores (misma lógica que antes)
# ---------------------------------------------------------------------------
modelo = None
error_carga = None
try:
    modelo = cargar_modelo(RUTA_MODELO)
except FileNotFoundError:
    error_carga = (
        f"No se encontró el archivo del modelo en '{RUTA_MODELO}'. "
        "Ejecuta primero el notebook 'notebooks/modelos_mineria_datos.ipynb' "
        "completo: la última celda guarda 'models/modelo_final.pkl'."
    )
except Exception as e:
    error_carga = f"Ocurrió un error al cargar el modelo: {e}"

# ---------------------------------------------------------------------------
# Hero / encabezado
# ---------------------------------------------------------------------------
render_hero(
    titulo="🚢 Predicción de Supervivencia — Titanic",
    subtitulo=(
        "Estima si un pasajero habría sobrevivido al naufragio del Titanic, a partir de "
        "sus características, usando un modelo Random Forest optimizado con GridSearchCV."
    ),
    eyebrow="Proyecto de Minería de Datos · Analítica de Datos · UPB",
    badges=(
        ["🌲 Random Forest", "🎯 GridSearchCV", "📊 891 pasajeros históricos", "✅ Validación cruzada 5-fold"]
        + (["🟢 Modelo cargado"] if error_carga is None else ["🔴 Modelo no disponible"])
    ),
)

if error_carga:
    st.error(error_carga)
    st.stop()
else:
    # Aviso breve (no intrusivo) de que el modelo quedó cargado correctamente;
    # reemplaza el antiguo `st.success("Modelo cargado correctamente.")` fijo
    # por una notificación tipo "toast", más propia de una app moderna.
    st.toast("Modelo cargado correctamente ✅", icon="✅")

# ---------------------------------------------------------------------------
# Cuerpo: formulario (izquierda) + panel informativo (derecha)
# ---------------------------------------------------------------------------
col_form, col_info = st.columns([1.6, 1], gap="large")

with col_form:
    with st.container(border=True):
        render_section_title("📝", "Datos del pasajero", "Completa el formulario y presiona \"Predecir\".")

        with st.form("formulario_prediccion"):
            col1, col2 = st.columns(2)

            with col1:
                pclass_label = st.selectbox(
                    "🎫 Clase del boleto",
                    options=["Primera clase", "Segunda clase", "Tercera clase"],
                    index=2,
                    help="Clase en la que viajaba el pasajero.",
                )
                sexo_label = st.selectbox(
                    "👤 Sexo",
                    options=["Mujer", "Hombre"],
                    help="Sexo del pasajero, tal como fue registrado en los datos históricos.",
                )
                edad = st.number_input(
                    "🎂 Edad (años)",
                    min_value=0.0,
                    max_value=100.0,
                    value=30.0,
                    step=1.0,
                    help="Edad del pasajero, entre 0 y 100 años.",
                )

            with col2:
                hermanos_esposos = st.number_input(
                    "👨‍👩‍👧 Hermanos/as o cónyuge a bordo",
                    min_value=0,
                    max_value=8,
                    value=0,
                    step=1,
                    help="Número de hermanos, hermanas o cónyuge a bordo (SibSp).",
                )
                padres_hijos = st.number_input(
                    "👶 Padres o hijos a bordo",
                    min_value=0,
                    max_value=6,
                    value=0,
                    step=1,
                    help="Número de padres o hijos a bordo (Parch).",
                )
                embarked_label = st.selectbox(
                    "⚓ Puerto de embarque",
                    options=["Southampton", "Cherbourg", "Queenstown"],
                    help="Puerto en el que el pasajero subió al barco.",
                )

            enviado = st.form_submit_button("🔮 Predecir supervivencia", use_container_width=True)

    # -----------------------------------------------------------------------
    # Validación y traducción de las etiquetas a los valores que espera el
    # modelo (misma lógica que antes, sin cambios)
    # -----------------------------------------------------------------------
    MAPA_PCLASS = {"Primera clase": 1, "Segunda clase": 2, "Tercera clase": 3}
    MAPA_SEXO = {"Mujer": "female", "Hombre": "male"}
    MAPA_EMBARKED = {"Southampton": "S", "Cherbourg": "C", "Queenstown": "Q"}
    # Se quitó el campo de tarifa del formulario (era confuso para el usuario:
    # nadie sabe cuánto costaba un boleto en libras de 1912). En su lugar, se
    # usa el promedio histórico real de tarifa por clase, calculado sobre el
    # propio dataset (ver data/titanic.csv): así el modelo -que sí necesita la
    # variable Fare, es una de sus 7 variables predictoras- sigue recibiendo un
    # valor realista y representativo de la clase elegida, sin pedírselo al
    # usuario.
    FARE_PROMEDIO_POR_CLASE = {
        "Primera clase": 84.15,
        "Segunda clase": 20.66,
        "Tercera clase": 13.68,
    }

    if enviado:
        errores = []
        if not (0 <= edad <= 100):
            errores.append("La edad debe estar entre 0 y 100 años.")
        if hermanos_esposos < 0 or padres_hijos < 0:
            errores.append("El número de familiares a bordo no puede ser negativo.")

        if errores:
            for e in errores:
                st.error(f"⚠️ {e}")
        else:
            try:
                tarifa = FARE_PROMEDIO_POR_CLASE[pclass_label]
                entrada = pd.DataFrame([{
                    "Pclass": MAPA_PCLASS[pclass_label],
                    "Sex": MAPA_SEXO[sexo_label],
                    "Age": float(edad),
                    "SibSp": int(hermanos_esposos),
                    "Parch": int(padres_hijos),
                    "Fare": float(tarifa),
                    "Embarked": MAPA_EMBARKED[embarked_label],
                }])

                prediccion = modelo.predict(entrada)[0]
                probabilidad = modelo.predict_proba(entrada)[0]
                prob_sobrevive = probabilidad[1]
                prob_no_sobrevive = probabilidad[0]

                st.markdown("<br>", unsafe_allow_html=True)
                render_section_title("🔮", "Resultado de la predicción")

                render_result_card(sobrevive=(prediccion == 1), probabilidad=prob_sobrevive)

                with st.container(border=True):
                    render_probability_bars(prob_sobrevive, prob_no_sobrevive)

                    with st.expander("🔍 Ver datos enviados al modelo"):
                        st.caption(
                            "La tarifa (Fare) no se pide en el formulario: se usa el "
                            "promedio histórico real de la clase elegida."
                        )
                        st.dataframe(entrada, hide_index=True, use_container_width=True)

            except Exception as e:
                st.error(f"❌ No fue posible realizar la predicción: {e}")

with col_info:
    with st.container(border=True):
        render_section_title("ℹ️", "Sobre el dataset")
        render_stat_row([
            ("891", "Pasajeros"),
            ("7", "Variables usadas"),
            ("38.4%", "Tasa de supervivencia"),
        ])
        st.markdown(
            '<p class="section-subtitle" style="margin-top:0.9rem;">'
            "Dataset histórico del Titanic, el mismo usado en las prácticas "
            "anteriores del equipo. Ver <code>data/README.md</code> para el "
            "detalle de cada columna."
            "</p>",
            unsafe_allow_html=True,
        )

    st.markdown("<div style='height:1rem'></div>", unsafe_allow_html=True)

    with st.container(border=True):
        render_section_title("🧠", "Sobre el modelo")
        render_stat_row([
            ("0.82", "Accuracy (CV)"),
            ("0.87", "ROC-AUC (CV)"),
        ])
        st.markdown(
            '<p class="section-subtitle" style="margin-top:0.9rem;">'
            "Random Forest optimizado con GridSearchCV sobre "
            "<code>n_estimators</code>, <code>max_depth</code>, "
            "<code>min_samples_leaf</code> y <code>max_features</code>, con "
            "validación cruzada estratificada de 5 particiones."
            "</p>",
            unsafe_allow_html=True,
        )

# ---------------------------------------------------------------------------
# Barra lateral: información del modelo (misma información que antes, con
# una presentación más cuidada)
# ---------------------------------------------------------------------------
with st.sidebar:
    st.markdown(
        '<h2 style="color:#f8fafc; font-size:1.3rem; margin-bottom:0.3rem;">🚢 Panel del modelo</h2>',
        unsafe_allow_html=True,
    )
    st.markdown(
        '<p style="color:#94a3b8; font-size:0.85rem; margin-top:0;">'
        "Resumen técnico del modelo desplegado."
        "</p>",
        unsafe_allow_html=True,
    )

    render_sidebar_card_start("Acerca del modelo", "🌲")
    render_sidebar_badges(["Random Forest", "GridSearchCV", "CV 5-fold"])
    st.markdown(
        """
        <div class="sb-fact">🎯 <b>Algoritmo:</b> Random Forest</div>
        <div class="sb-fact">⚙️ <b>Optimización:</b> GridSearchCV (validación cruzada de 5 particiones)</div>
        <div class="sb-fact">📋 <b>Variables:</b> Clase, Sexo, Edad, SibSp, Parch, Tarifa (estimada por clase), Puerto de embarque</div>
        <div class="sb-fact">💾 <b>Dataset:</b> Titanic (891 pasajeros)</div>
        """,
        unsafe_allow_html=True,
    )
    render_sidebar_card_end()

    try:
        prep = modelo.named_steps["prep"]
        clf = modelo.named_steps["clf"]
        nombres = prep.get_feature_names_out()
        importancias = pd.Series(clf.feature_importances_, index=nombres).sort_values(ascending=False)

        render_sidebar_card_start("Importancia de variables", "📊")
        fig = render_feature_importance_chart(importancias)
        st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})
        render_sidebar_card_end()
    except Exception:
        pass

    st.markdown(
        '<p style="color:#64748b; font-size:0.76rem; margin-top:1rem;">'
        "⚠️ Modelo con fines académicos, entrenado sobre datos históricos del "
        "Titanic. No debe usarse para ningún propósito real."
        "</p>",
        unsafe_allow_html=True,
    )


Overwriting ../app/despliegue.py


In [8]:
# Verificación: confirmamos que los 4 archivos quedaron escritos
archivos_generados = [
    '../.streamlit/config.toml',
    '../app/assets/style.css',
    '../app/components.py',
    '../app/despliegue.py',
]
for ruta in archivos_generados:
    existe = os.path.exists(ruta)
    tam = os.path.getsize(ruta) if existe else 0
    print(f"{ruta:30s} existe={existe}  ({tam} bytes)")

../.streamlit/config.toml      existe=True  (155 bytes)
../app/assets/style.css        existe=True  (11250 bytes)
../app/components.py           existe=True  (5684 bytes)
../app/despliegue.py           existe=True  (12674 bytes)


## 4. Cómo iniciar la aplicación

**Desde la terminal (recomendado), parado en la carpeta raíz del proyecto:**

```bash
pip install -r requirements.txt
streamlit run app/despliegue.py
```

Streamlit abrirá automáticamente una pestaña del navegador en `http://localhost:8501`. Si no se abre sola, copia esa dirección y pégala en el navegador.

**Desde Visual Studio Code:**

1. Abre la carpeta del proyecto en VS Code (`File > Open Folder`).
2. Abre una terminal integrada (`Terminal > New Terminal`).
3. Activa el entorno virtual si usas uno (`.venv\Scripts\activate` en Windows, o el que tengas configurado).
4. Ejecuta `pip install -r requirements.txt` (solo la primera vez, o si cambiaron las dependencias).
5. Ejecuta `streamlit run app/despliegue.py`.
6. VS Code puede preguntarte si quieres abrir el puerto en el navegador — acepta, o entra manualmente a `http://localhost:8501`.

Para detener la aplicación, vuelve a la terminal y presiona `Ctrl+C`.

## 5. Cómo tomar la captura de pantalla que pide el profesor

1. Inicia la aplicación con `streamlit run app/despliegue.py` y espera a que se abra en el navegador.
2. Llena el formulario con un pasajero de ejemplo (por ejemplo, el que usamos arriba: primera clase, mujer, 29 años, 0 familiares, tarifa 80, Southampton).
3. Haz clic en **"Predecir supervivencia"** y espera a que aparezca el resultado (la sección "Resultado" con la predicción y las probabilidades).
4. Toma la captura de pantalla completa de la ventana del navegador, de forma que se vea: el formulario con los datos ingresados, el resultado de la predicción y, si quieres, la barra lateral con la importancia de variables.
   - En Windows: `Win + Shift + S` para recortar solo la ventana, o `Alt + Impr Pant` para capturar la ventana activa completa.
5. Guarda la imagen (por ejemplo como `captura_despliegue.png`) y adjúntala en la tarea de Teams junto con los notebooks, tal como pide el profesor.